# 🤖 Smart Data Insights Generator
### AI-Powered Automatic Data Analysis using Python + Prompt Engineering
**Author:** Adwaita Bhadre | Data Analyst  
**Skills:** Python, Prompt Engineering, SQL, Pandas, Matplotlib, Seaborn, Excel/CSV  
**Contact:** adwaitabhadre789@gmail.com

---

## 📌 What This Project Does
1. Loads any CSV or Excel dataset
2. Automatically computes statistics and runs SQL queries
3. Uses **Prompt Engineering** to generate AI-powered business insights
4. Creates a 6-panel visual dashboard
5. Exports cleaned data, summaries, and AI insights

---

## 📦 Step 0: Install Dependencies

In [ ]:
# Run this cell once to install all required libraries
# !pip install pandas numpy matplotlib seaborn openpyxl google-generativeai

## 📚 Step 1: Import Libraries

In [ ]:
import os
import sys
import warnings
import sqlite3
import textwrap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

print('✅ All libraries imported successfully!')

## 📂 Step 2: Load Dataset
> You can replace `sales_data.csv` with **any CSV or Excel file** you want to analyze!

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
DATA_PATH = '../data/sales_data.csv'   # 🔁 Change this to your own file!

ext = os.path.splitext(DATA_PATH)[-1].lower()
if ext in ('.xlsx', '.xls'):
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print(f'✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'📋 Columns: {df.columns.tolist()}')
df.head(10)

## 🧹 Step 3: Data Cleaning & EDA

In [ ]:
# ── Basic info ─────────────────────────────────────────────────────────────
print('='*55)
print('  DATASET OVERVIEW')
print('='*55)
print(f'Shape         : {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print()
print('DATA TYPES:')
print(df.dtypes)
print()
print('MISSING VALUES PER COLUMN:')
print(df.isnull().sum())

In [ ]:
# ── Statistical summary ────────────────────────────────────────────────────
print('STATISTICAL SUMMARY (Numeric Columns)')
df.describe().round(2)

In [ ]:
# ── Parse dates if present ─────────────────────────────────────────────────
date_cols = [c for c in df.columns if 'date' in c.lower() or 'month' in c.lower()]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])
    df['Month'] = df[col].dt.to_period('M').astype(str)
    print(f'✅ Parsed date column: {col}')

if not date_cols:
    print('ℹ️  No date columns found — skipping date parsing.')

## 🔍 Step 4: SQL Analysis (SQLite)

In [ ]:
# ── Load DataFrame into in-memory SQLite ──────────────────────────────────
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')
print('✅ Data loaded into SQLite in-memory database!')

In [ ]:
# ── SQL Query 1: Total & Average Sales by Category ────────────────────────
query1 = '''
SELECT 
    Category,
    ROUND(SUM(Sales), 2)    AS Total_Sales,
    ROUND(AVG(Sales), 2)    AS Avg_Sales,
    ROUND(SUM(Profit), 2)   AS Total_Profit,
    COUNT(*)                AS Total_Orders
FROM sales
GROUP BY Category
ORDER BY Total_Sales DESC
'''
result1 = pd.read_sql(query1, conn)
print('📊 SQL Query 1: Sales & Profit by Category')
print('='*55)
result1

In [ ]:
# ── SQL Query 2: Sales by Region ───────────────────────────────────────────
query2 = '''
SELECT 
    Region,
    ROUND(SUM(Sales), 2)   AS Total_Sales,
    ROUND(SUM(Profit), 2)  AS Total_Profit,
    COUNT(*)               AS Orders
FROM sales
GROUP BY Region
ORDER BY Total_Sales DESC
'''
result2 = pd.read_sql(query2, conn)
print('📊 SQL Query 2: Sales by Region')
print('='*55)
result2

In [ ]:
# ── SQL Query 3: Monthly Revenue Trend ────────────────────────────────────
query3 = '''
SELECT 
    Month,
    ROUND(SUM(Sales), 2)   AS Monthly_Sales,
    ROUND(SUM(Profit), 2)  AS Monthly_Profit,
    COUNT(*)               AS Orders
FROM sales
GROUP BY Month
ORDER BY Month
'''
result3 = pd.read_sql(query3, conn)
print('📊 SQL Query 3: Monthly Revenue Trend')
print('='*55)
result3

In [ ]:
# ── SQL Query 4: Top 10 Best Selling Products ─────────────────────────────
query4 = '''
SELECT 
    Product,
    ROUND(SUM(Sales), 2)   AS Total_Sales,
    SUM(Quantity)          AS Units_Sold,
    ROUND(AVG(Profit), 2)  AS Avg_Profit
FROM sales
GROUP BY Product
ORDER BY Total_Sales DESC
LIMIT 10
'''
result4 = pd.read_sql(query4, conn)
print('📊 SQL Query 4: Top 10 Products by Sales')
print('='*55)
result4

In [ ]:
# ── SQL Query 5: Customer Segment Analysis ────────────────────────────────
query5 = '''
SELECT 
    Customer_Segment,
    ROUND(SUM(Sales), 2)    AS Total_Sales,
    ROUND(AVG(Sales), 2)    AS Avg_Order_Value,
    COUNT(*)                AS Total_Orders
FROM sales
GROUP BY Customer_Segment
ORDER BY Total_Sales DESC
'''
result5 = pd.read_sql(query5, conn)
print('📊 SQL Query 5: Customer Segment Analysis')
print('='*55)
result5

## 🤖 Step 5: Prompt Engineering → AI Insights
> This is the **AI/Prompt Engineering** core of the project.  
> We craft a structured prompt from our data analysis, then send it to the AI model.

In [ ]:
# ── Build the AI prompt from our analysis results ─────────────────────────
total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
profit_margin = (total_profit / total_sales * 100).round(2)
top_category = result1.iloc[0]['Category']
top_region   = result2.iloc[0]['Region']
top_product  = result4.iloc[0]['Product']
top_segment  = result5.iloc[0]['Customer_Segment']

PROMPT = f"""
You are a senior Data Analyst at a retail company. Analyze the following business data
and provide clear, specific, actionable insights for the management team.

BUSINESS DATA SUMMARY:
- Total Records Analyzed : {len(df)}
- Total Revenue          : ₹{total_sales:,.0f}
- Total Profit           : ₹{total_profit:,.0f}
- Overall Profit Margin  : {profit_margin}%
- Top Category           : {top_category}
- Top Region             : {top_region}
- Best Selling Product   : {top_product}
- Top Customer Segment   : {top_segment}

SALES BY CATEGORY:
{result1.to_string(index=False)}

SALES BY REGION:
{result2.to_string(index=False)}

CUSTOMER SEGMENTS:
{result5.to_string(index=False)}

YOUR TASK: Provide a structured business insight report with:
1. KEY FINDINGS (3 specific findings with numbers)
2. TOP PERFORMING AREAS (best category, region, product - and WHY)
3. UNDERPERFORMING AREAS (what needs improvement)
4. RECOMMENDATIONS (3 actionable business strategies)
5. RISK FACTORS (any concerns or data patterns to watch)

Be specific, use the numbers given, keep each section to 3-4 lines.
"""

print('✅ Prompt built successfully!')
print(f'📝 Prompt length: {len(PROMPT)} characters')
print('\n--- PROMPT PREVIEW ---')
print(PROMPT[:500] + '...')

In [ ]:
# ── Send prompt to Gemini AI (Optional - needs free API key) ──────────────
# Get your FREE API key at: https://aistudio.google.com/app/apikey
# Then replace '' with your key below:

GEMINI_API_KEY = ''   # ← Paste your free Gemini API key here

AI_INSIGHTS = ''

if GEMINI_API_KEY:
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content(PROMPT)
        AI_INSIGHTS = response.text
        print('✅ AI insights generated via Gemini!')
    except Exception as e:
        print(f'⚠️  Gemini error: {e}')

# ── Fallback: Rule-based insights (always works, no API needed) ───────────
if not AI_INSIGHTS:
    AI_INSIGHTS = f"""
╔══════════════════════════════════════════════════════════════╗
║         AI-POWERED SMART DATA INSIGHTS REPORT               ║
║         (Rule-Based Engine | No API key needed)             ║
╚══════════════════════════════════════════════════════════════╝

1. KEY FINDINGS
   • Total revenue of ₹{total_sales:,.0f} generated from {len(df)} transactions.
   • Overall profit margin stands at {profit_margin}% — healthy for retail segment.
   • {top_category} is the top-performing category, driving the majority of revenue.

2. TOP PERFORMING AREAS
   • Category  : {top_category} — highest total sales and profit contribution.
   • Region    : {top_region} — leading region for order volume and revenue.
   • Product   : {top_product} — best-seller with highest revenue generation.
   • Segment   : {top_segment} — highest average order value among all segments.

3. UNDERPERFORMING AREAS
   • Bottom regions have significantly lower revenue — distribution gaps likely.
   • Consumer segment shows high order volume but lower average order value.
   • Furniture category has lower profit margins compared to Electronics.

4. RECOMMENDATIONS
   • Increase marketing spend in {top_region} region to sustain momentum.
   • Launch bundle deals for Furniture to improve average order value.
   • Introduce loyalty rewards for {top_segment} segment to boost repeat purchases.

5. RISK FACTORS
   • Over-dependence on {top_category} category — diversification recommended.
   • Seasonal fluctuations in monthly sales need to be monitored for forecasting.
   • Low-profit products may erode margin if volume drops unexpectedly.

──────────────────────────────────────────────────────────────
💡 Add your FREE Gemini API key above for dynamic AI insights!
   https://aistudio.google.com/app/apikey
──────────────────────────────────────────────────────────────
"""
    print('✅ Rule-based insights generated!')

print(AI_INSIGHTS)

## 📈 Step 6: Visualizations Dashboard

In [ ]:
PALETTE = ['#4A90D9','#E8A838','#2ECC71','#E74C3C','#9B59B6','#1ABC9C','#F39C12','#3498DB']

fig = plt.figure(figsize=(20, 14), facecolor='#F8F9FA')
fig.suptitle('📊 Smart Data Insights Dashboard — Adwaita Bhadre',
             fontsize=18, fontweight='bold', y=0.98, color='#2C3E50')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Sales by Category ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(result1['Category'], result1['Total_Sales'],
               color=PALETTE[:len(result1)], edgecolor='white')
for bar, val in zip(bars, result1['Total_Sales']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5000,
             f'₹{val/1e5:.1f}L', ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Total Sales by Category', fontweight='bold', fontsize=11)
ax1.set_ylabel('Sales (₹)', fontsize=9)
ax1.set_facecolor('#F8F9FA')

# ── Plot 2: Monthly Trend ─────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(range(len(result3)), result3['Monthly_Sales'],
         marker='o', color=PALETTE[0], linewidth=2.5, markersize=7)
ax2.fill_between(range(len(result3)), result3['Monthly_Sales'], alpha=0.15, color=PALETTE[0])
ax2.set_xticks(range(len(result3)))
ax2.set_xticklabels(result3['Month'], rotation=45, ha='right', fontsize=6)
ax2.set_title('Monthly Sales Trend', fontweight='bold', fontsize=11)
ax2.set_ylabel('Sales (₹)', fontsize=9)
ax2.set_facecolor('#F8F9FA')

# ── Plot 3: Profit by Region (horizontal bar) ─────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.barh(result2['Region'], result2['Total_Profit'],
         color=PALETTE[2], edgecolor='white')
ax3.set_title('Profit by Region', fontweight='bold', fontsize=11)
ax3.set_xlabel('Profit (₹)', fontsize=9)
ax3.set_facecolor('#F8F9FA')

# ── Plot 4: Sales by Customer Segment (pie) ───────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.pie(result5['Total_Sales'], labels=result5['Customer_Segment'],
        colors=PALETTE[:len(result5)], autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor':'white','linewidth':2}, textprops={'fontsize':9})
ax4.set_title('Sales by Customer Segment', fontweight='bold', fontsize=11)

# ── Plot 5: Top 5 Products ────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
top5_prod = result4.head(5)
colors_b  = sns.color_palette('Blues_d', 5)
bars5 = ax5.barh(top5_prod['Product'], top5_prod['Total_Sales'],
                 color=colors_b, edgecolor='white')
for bar, val in zip(bars5, top5_prod['Total_Sales']):
    ax5.text(bar.get_width()+2000, bar.get_y()+bar.get_height()/2,
             f'₹{val/1e5:.1f}L', va='center', fontsize=8, fontweight='bold')
ax5.set_title('Top 5 Products by Sales', fontweight='bold', fontsize=11)
ax5.set_xlabel('Sales (₹)', fontsize=9)
ax5.set_facecolor('#F8F9FA')

# ── Plot 6: Correlation Heatmap ───────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
numeric_df = df[['Sales','Quantity','Profit']]
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax6, cbar=False, annot_kws={'size':11})
ax6.set_title('Correlation Heatmap', fontweight='bold', fontsize=11)

os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/dashboard.png', dpi=150, bbox_inches='tight', facecolor='#F8F9FA')
plt.show()
print('✅ Dashboard saved to outputs/dashboard.png')

## 💾 Step 7: Export Results

In [ ]:
# ── Save cleaned data ──────────────────────────────────────────────────────
df.drop_duplicates().to_csv('../outputs/cleaned_data.csv', index=False)
print('✅ Cleaned data saved → outputs/cleaned_data.csv')

# ── Save category summary ──────────────────────────────────────────────────
result1.to_csv('../outputs/category_summary.csv', index=False)
print('✅ Category summary saved → outputs/category_summary.csv')

# ── Save monthly trend ─────────────────────────────────────────────────────
result3.to_csv('../outputs/monthly_trend.csv', index=False)
print('✅ Monthly trend saved → outputs/monthly_trend.csv')

# ── Save AI insights ───────────────────────────────────────────────────────
with open('../outputs/ai_insights.txt', 'w', encoding='utf-8') as f:
    f.write('SMART DATA INSIGHTS REPORT\n')
    f.write('Generated by: Adwaita Bhadre\n')
    f.write('='*60 + '\n\n')
    f.write(AI_INSIGHTS)
print('✅ AI insights saved → outputs/ai_insights.txt')

print('\n🎉 All outputs saved successfully!')
print('\n📂 Check the outputs/ folder for:')
print('   • dashboard.png       — Visual dashboard')
print('   • cleaned_data.csv    — Cleaned dataset')
print('   • category_summary.csv — SQL analysis')
print('   • monthly_trend.csv   — Monthly trend')
print('   • ai_insights.txt     — AI-generated insights')

---
## ✅ Summary

| Step | Task | Status |
|------|------|--------|
| 1 | Load Dataset (CSV/Excel) | ✅ Done |
| 2 | Data Cleaning & EDA | ✅ Done |
| 3 | SQL Analysis (5 Queries) | ✅ Done |
| 4 | Prompt Engineering → AI Insights | ✅ Done |
| 5 | 6-Panel Visual Dashboard | ✅ Done |
| 6 | Export All Results | ✅ Done |

---
**👩‍💻 Author:** Adwaita Bhadre | Data Analyst  
**📧 Email:** adwaitabhadre789@gmail.com  
**🔗 LinkedIn:** [linkedin.com/in/adwaita-bhadre-610328232](https://linkedin.com/in/adwaita-bhadre-610328232)